In [ ]:
                #-----Dataset & Dataloader--------#
# Datset and Dataloder are core abstructions in PyTorch.
# It is use to decouple how you define your data from how you efficiently iterate over it in training loops


In [ ]:
                # ------Dataset Class-----#
# The dataset class is essentially a bluprint
# When you create a custom Dataset, you decide how data is loaded and returned.
# Dataset class load data/Image one by one from memory where data stored.

# It defines:
    # __init__(): tells how data should be loaded
    # __len__(): returns the total number of samples
    # __getitem__(index): return the data (and label) at the given index


In [ ]:
# Structure of Dataset Class
from torch.utils.data import Dataset
import torch

class MyDataset(Dataset):

    def __init__(self):                 # Stores the data and labels when the dataset object is created.
        self.features = torch.tensor([
            [1.0, 2.0],
            [3.0, 4.0],
            [5.0, 6.0]
        ])

        self.labels = torch.tensor([0, 1, 0])

    def __len__(self):                  # Returns the total number of samples.
        return len(self.features)

    def __getitem__(self, index):       # Returns one sample and its label.
        return self.features[index], self.labels[index]

In [ ]:
            #--------DataLoader---------#
# The Dataloader wraps a Dataset and handles batching, shuffling, and parallel loading for you.
# A DataLoader takes a Dataset and provides an iterable that yields batches of data.

In [ ]:
# DataLoader Control Flow
    # At the start of each epoch the dataloader (if shuffle true) shuffles indices using sampler.
    # It devides the indices into chunks of batch_size
    # for each index in the chunk data samples are fetched from the dataset object

In [ ]:
# Structure of dataloader Class
from torch.utils.data import DataLoader

dataset = MyDataset()

dataloader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True
)

In [5]:
# Code Example

from sklearn.datasets import make_classification
from torch.utils.data import Dataset, DataLoader
import torch

In [3]:
# Step 1: Create a synthetic classification dataset using sklearn
X, y = make_classification(
    n_samples=10,       # Number of samples
    n_features=2,       # Number of features
    n_informative=2,    # Number of informative features
    n_redundant=0,      # Number of redundant features
    n_classes=2,        # Number of classes
    random_state=42     # For reproducibility
)

In [4]:
# Convert the data to PyTorch tensors
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

In [6]:
# Creates custom dataset
class CustomDataset(Dataset):

  def __init__(self, features, labels):

    self.features = features
    self.labels = labels

  def __len__(self):

    return self.features.shape[0]

  def __getitem__(self, index):

    return self.features[index], self.labels[index]

In [8]:
# Create object
dataset=CustomDataset(X,y)
len(dataset)

10

In [9]:
# Creates Dataloader object
dataloader = DataLoader(dataset, batch_size=2, shuffle=False)

In [10]:
for batch_features, batch_labels in dataloader:

  print(batch_features)
  print(batch_labels)
  print("-"*50)

tensor([[ 1.0683, -0.9701],
        [-1.1402, -0.8388]])
tensor([1, 0])
--------------------------------------------------
tensor([[-2.8954,  1.9769],
        [-0.7206, -0.9606]])
tensor([0, 0])
--------------------------------------------------
tensor([[-1.9629, -0.9923],
        [-0.9382, -0.5430]])
tensor([0, 1])
--------------------------------------------------
tensor([[ 1.7273, -1.1858],
        [ 1.7774,  1.5116]])
tensor([1, 1])
--------------------------------------------------
tensor([[ 1.8997,  0.8344],
        [-0.5872, -1.9717]])
tensor([1, 0])
--------------------------------------------------


In [ ]:
        # -------------Sampler-----------#
# In PyTorch the sampler in the dataloder determines the strategy for selecting samples from the dataset during Data Loading.
# It controls how indices of the dataset are drawn for each batch.
# It has predefined sampler

# SequentialSampler:
    # Samples element sequentially, in the order they apear in the dataset.
    # Default when shuffle=False
# RandomSampler:
    # Samples element randomely without replacement
    # Default when shuffle=True

In [ ]:
            #--------collate_fn---------# 

# collate_fn function in PyTorch's Dataloader is a function that specifies how to combine a list of samples from a dataset into a single batch.


In [ ]:
# Important Parameter in Dataloader
# 1. Dataset(Mandatory)
    # The dataset from which dataloader will pull data
# 2. batch_size:
    # How many samples per batch to load
    # Default=1
    # Larger batch sizes can speed up training on gpu
# 3. shuffle
    # If true shuffle the indices of dataset
# 4. num_workers: 
    # Used to load data in parallel
# 5. Pin memory
    # improve GPU transfer speed
# 6. collate_fn
    # process a list of sample into batch
# 7. sampler
    # provides strategy to draw samples


In [11]:
# Improving the cancer classification code
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [ ]:
# Load Dataset
df=pd.read_csv('breast_cancer.csv')

In [ ]:
# Drop unnecessary features
df.drop(columns=['id'], inplace= True)

In [15]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [ ]:
# Split the data into train test set
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

In [ ]:
# Perform scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# Perform label encoding
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [ ]:
# Convert array to tensor
X_train_tensor = torch.from_numpy(X_train.astype(np.float32))
X_test_tensor = torch.from_numpy(X_test.astype(np.float32))
y_train_tensor = torch.from_numpy(y_train.astype(np.float32))
y_test_tensor = torch.from_numpy(y_test.astype(np.float32))

In [ ]:
# Create custom dataset class

from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):

  def __init__(self, features, labels):

    self.features = features
    self.labels = labels

  def __len__(self):

    return len(self.features)

  def __getitem__(self, idx):

    return self.features[idx], self.labels[idx]

In [ ]:
# Creates Dataloader
train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)

In [ ]:
# Load Data
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [ ]:
# Model class
import torch.nn as nn


class MySimpleNN(nn.Module):

  def __init__(self, num_features):

    super().__init__()
    self.linear = nn.Linear(num_features, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, features):

    out = self.linear(features)
    out = self.sigmoid(out)

    return out

In [ ]:
# Parameters
learning_rate = 0.1
epochs = 25

In [25]:
# create model
model = MySimpleNN(X_train_tensor.shape[1])

# define optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# define loss function
loss_function = nn.BCELoss()

In [26]:

# define loop
for epoch in range(epochs):

  for batch_features, batch_labels in train_loader:

    # forward pass
    y_pred = model(batch_features)

    # loss calculate
    loss = loss_function(y_pred, batch_labels.view(-1,1))

    # clear gradients
    optimizer.zero_grad()

    # backward pass
    loss.backward()

    # parameters update
    optimizer.step()

  # print loss in each epoch
  print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 0.11228961497545242
Epoch: 2, Loss: 0.10572566837072372
Epoch: 3, Loss: 0.08312834799289703
Epoch: 4, Loss: 0.05781231075525284
Epoch: 5, Loss: 0.10564209520816803
Epoch: 6, Loss: 0.047043103724718094
Epoch: 7, Loss: 0.12435362488031387
Epoch: 8, Loss: 0.004017477855086327
Epoch: 9, Loss: 0.03631569817662239
Epoch: 10, Loss: 0.02621982991695404
Epoch: 11, Loss: 0.12004499137401581
Epoch: 12, Loss: 0.01497791800647974
Epoch: 13, Loss: 0.010360288433730602
Epoch: 14, Loss: 0.09709997475147247
Epoch: 15, Loss: 0.07849705219268799
Epoch: 16, Loss: 0.030114589259028435
Epoch: 17, Loss: 0.02589665912091732
Epoch: 18, Loss: 0.16499851644039154
Epoch: 19, Loss: 0.1530594676733017
Epoch: 20, Loss: 0.03220553696155548
Epoch: 21, Loss: 0.06277237087488174
Epoch: 22, Loss: 0.11751759797334671
Epoch: 23, Loss: 0.022919628769159317
Epoch: 24, Loss: 0.010533662512898445
Epoch: 25, Loss: 0.004018464591354132


In [27]:
# Model evaluation using test_loader
model.eval()  # Set the model to evaluation mode
accuracy_list = []

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        # Forward pass
        y_pred = model(batch_features)
        y_pred = (y_pred > 0.8).float()  # Convert probabilities to binary predictions

        # Calculate accuracy for the current batch
        batch_accuracy = (y_pred.view(-1) == batch_labels).float().mean().item()
        accuracy_list.append(batch_accuracy)

# Calculate overall accuracy
overall_accuracy = sum(accuracy_list) / len(accuracy_list)
print(f'Accuracy: {overall_accuracy:.4f}')

Accuracy: 0.9470
